# Build candidates (repo)

In [1]:
from pathlib import Path
import sys
from pyspark.sql import functions as sf

In [6]:
# Make repo root importable regardless of Jupyter working directory
root = Path.cwd()
while root != root.parent and not (root / "pyproject.toml").exists():
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from notebooks.spark_session import create_spark_session

spark = create_spark_session('build-repo-candidates')
spark

In [7]:
print('master:', spark.sparkContext.master)
print('app:', spark.sparkContext.appName)

master: spark://spark-master:7077
app: build-repo-candidates


In [8]:
path = "s3a://gba-silver-zone-prod/repo_aggregates/dt=2026-03-28/hr=19/"
df = spark.read.parquet(path)
df.select("*").distinct().show(5, truncate=False)

26/03/29 20:51:40 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
26/03/29 20:51:43 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

+---------+---------------------------+------------+-------------------+-------------+-------------------+----------+------------+--------------+--------------+---------------+--------------------------+-----------+-------------+--------------+---------------------+-------------+----------------------------------+--------------------+-------------+-------------+-----------+-------------+-------------+------------+-------------------+-----------------+--------------------------------+-----------------+-------------------+--------------------+---------------------------+-------------------+----------------------------------------+--------------------------+-------------------+-------------------+-----------------+-------------------+-------------------+------------------+-------------------------+-----------------------+-------------------+---------+----------+---+
|repo_id  |repo_full_name             |total_events|public_events_count|unique_actors|last_event_at      |bot_events|issues_

In [9]:
METRICS = ['total_events', 'public_events_count']

In [10]:
frames = []

In [14]:
from pyspark.sql import functions as F
from pyspark.sql import Window

In [18]:
ranked = df.filter(F.isnotnull(F.col('total_events')))
ranked.select('repo_id', 'repo_full_name', 'total_events').show()

+-------+--------------------+------------+
|repo_id|      repo_full_name|total_events|
+-------+--------------------+------------+
|    130|   ruby-git/ruby-git|           5|
|   1734|           aasm/aasm|           1|
|  47476|         qtile/qtile|           1|
| 101248| samuelclay/NewsBlur|           1|
| 108110|       mongodb/mongo|           2|
| 143131|     danmar/cppcheck|           4|
| 204362|maxmaximov/.dotfiles|           1|
| 206387|     apache/activemq|           1|
| 302322|       gradle/gradle|          13|
| 317887|liferay/liferay-p...|           2|
| 458058|     symfony/symfony|           1|
| 507775|elastic/elasticse...|           1|
| 538746|           ruby/ruby|           1|
| 569041|           curl/curl|           1|
| 640534|         sympy/sympy|           3|
| 802525|         sfepy/sfepy|           1|
| 812033|     gabebw/dotfiles|           1|
| 839336|simplecov-ruby/si...|           1|
| 858127|   pandas-dev/pandas|           8|
| 873328|    getsentry/sentry|  

In [23]:
ranked = ranked.withColumn("metrik_rank", F.row_number().over(
    Window.orderBy(F.desc_nulls_last(F.col('total_events')))
))
ranked.select('repo_id', 'repo_full_name', 'total_events', 'metrik_rank').show()

26/03/29 21:23:02 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/29 21:23:02 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/29 21:23:02 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+----------+--------------------+------------+-----------+
|   repo_id|      repo_full_name|total_events|metrik_rank|
+----------+--------------------+------------+-----------+
|1191714265|0williams-johng/g...|        2313|          1|
|1178335935|etsy-images/etsy-...|        1329|          2|
|1191186263|        howallinna/-|         613|          3|
|1188536554| bhang07879/culrrhju|         492|          4|
|1125932955|       Carbare/teste|         441|          5|
| 782549802|StefanMaron/MSDyn...|         425|          6|
|1170163669|dipsylala/polymar...|         412|          7|
|1102137324|nekovach-commits/...|         403|          8|
|1076470140|gaston1799/RobloxLua|         381|          9|
|1010205745|Assist071/MyLoade...|         380|         10|
|1193428274|   DOCGroup/bugzilla|         380|         11|
|1176335208|    hyperspaceai/agi|         378|         12|
|1170717199|MinBZK/regelrecht...|         348|         13|
|1099385184|escapingwork/teen...|         340|         1

In [29]:
ranked = ranked.filter(
    F.col("metrik_rank") <= 10
).drop(
    "metrik_rank"
).withColumn(
    'selection_metric', F.lit('total_events')
)
ranked.select('repo_id', 'repo_full_name', 'total_events', 'selection_metric').show()

26/03/29 21:25:33 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/29 21:25:33 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/29 21:25:33 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+----------+--------------------+------------+----------------+
|   repo_id|      repo_full_name|total_events|selection_metric|
+----------+--------------------+------------+----------------+
|1191714265|0williams-johng/g...|        2313|    total_events|
|1178335935|etsy-images/etsy-...|        1329|    total_events|
|1191186263|        howallinna/-|         613|    total_events|
|1188536554| bhang07879/culrrhju|         492|    total_events|
|1125932955|       Carbare/teste|         441|    total_events|
| 782549802|StefanMaron/MSDyn...|         425|    total_events|
|1170163669|dipsylala/polymar...|         412|    total_events|
|1102137324|nekovach-commits/...|         403|    total_events|
|1076470140|gaston1799/RobloxLua|         381|    total_events|
|1193428274|   DOCGroup/bugzilla|         380|    total_events|
+----------+--------------------+------------+----------------+



In [30]:
ranked_1 = df.filter(F.isnotnull(F.col('public_events_count'))).withColumn("metrik_rank", F.row_number().over(
    Window.orderBy(F.desc_nulls_last(F.col('public_events_count')))
)).filter(
    F.col("metrik_rank") <= 10
).drop(
    "metrik_rank"
).withColumn(
    'selection_metric', F.lit('public_events_count')
)
ranked_1.select('repo_id', 'repo_full_name', 'public_events_count', 'selection_metric').show()

26/03/29 21:26:11 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/29 21:26:11 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/29 21:26:11 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
[Stage 22:>                                                         (0 + 2) / 2]

+----------+--------------------+-------------------+-------------------+
|   repo_id|      repo_full_name|public_events_count|   selection_metric|
+----------+--------------------+-------------------+-------------------+
|1191714265|0williams-johng/g...|               2313|public_events_count|
|1178335935|etsy-images/etsy-...|               1329|public_events_count|
|1191186263|        howallinna/-|                613|public_events_count|
|1188536554| bhang07879/culrrhju|                492|public_events_count|
|1125932955|       Carbare/teste|                441|public_events_count|
| 782549802|StefanMaron/MSDyn...|                425|public_events_count|
|1170163669|dipsylala/polymar...|                412|public_events_count|
|1102137324|nekovach-commits/...|                403|public_events_count|
|1076470140|gaston1799/RobloxLua|                381|public_events_count|
|1193428274|   DOCGroup/bugzilla|                380|public_events_count|
+----------+--------------------+-----

In [37]:
ranked_3 = ranked.unionByName(ranked_1)

In [47]:
ranked_final = ranked_3.groupBy('repo_id', 'repo_full_name').agg(
    *[F.first(c, ignorenulls=True).alias(c) for c in df.columns if c not in {'repo_id', 'repo_full_name'}],
    F.array_sort(F.collect_set("selection_metric")).alias("selection_reasons")
)
ranked_final.show()

26/03/29 21:49:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/29 21:49:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/29 21:49:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/03/29 21:49:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
[Stage 90:>                 (0 + 2) / 2][Stage 91:>                 (0 + 2) / 2]

+----------+--------------------+------------+-------------------+-------------+-------------------+----------+------------+--------------+--------------+---------------+--------------------------+-----------+-------------+--------------+---------------------+-------------+----------------------------------+--------------------+-------------+-------------+-----------+-------------+-------------+------------+-------------------+-----------------+--------------------------------+--------------------+-------------------+--------------------+---------------------------+-------------------+----------------------------------------+--------------------------+-------------------+-------------------+-----------------+-------------------+-------------------+------------------+-------------------------+-----------------------+------------------+---------+----------+---+--------------------+
|   repo_id|      repo_full_name|total_events|public_events_count|unique_actors|      last_event_at|bot_eve

In [48]:
spark.stop()